# 第 1 周末练习 —— 双后端技术问答（Groq 云端 + Ollama 本地）

## 练习目标（理念）

为了展示你对 **OpenAI 兼容 API**（云端 Groq）以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一段业务相关的技术问题 / 代码（这里是 Stayez 定价函数）
- **输出**：清晰的 Markdown 解释
- **对比**：同一问题分别问云端大模型与本地小模型

这是你在课程期间自己也能天天用的工具。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| 同一 SDK、不同 `base_url` | Groq 与 Ollama 都用 `OpenAI(...)` |
| 流式 vs 非流式 | Groq 用 `stream=True`；本地一次取完整 `message.content` |
| System Prompt | 统一的 `SYSTEM_PROMPT` 约束解释风格 |

## 怎么跑

1. `.env` 里准备 `GROQ_API_KEY`；本机 Ollama 需已拉取 `gemma3:270m`（或改常量）
2. 从上到下运行；可改 `question` 再分别跑云端格与本地格做对比


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（例如 GROQ_API_KEY）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：Groq / Ollama 都走 OpenAI 兼容协议
from openai import OpenAI
# 从 IPython.display 导入 Markdown 与 display：在笔记本里渲染回答
from IPython.display import Markdown, display


In [2]:
# ========== 常量：两个后端的模型名 + 共用的 system prompt ==========

# Groq 上的云端模型 id（字符串必须与 Groq 控制台/文档中的模型名一致）
MODEL_CLOUD = 'llama-3.3-70b-versatile'  # Groq model
# 本地 Ollama 模型名：需事先 pull；可按本机已安装模型修改
MODEL_LOCAL = 'gemma3:270m'               # Ollama model

# 发给模型的系统指令：保留英文，规定「世界级技术专家 + Markdown 作答」
SYSTEM_PROMPT = """You are a world-class technical expert.
Explain the provided technical question or code snippet clearly.
Respond in Markdown."""


In [3]:
# ========== 环境 + 两个客户端：云端 Groq 与本地 Ollama ==========

# 加载 .env：override=True 表示用文件值覆盖已有环境变量
load_dotenv(override=True)

# Groq 客户端：base_url 指向 Groq 的 OpenAI 兼容端点；密钥从环境变量读取
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv('GROQ_API_KEY')
)

# Ollama 客户端：指向本机 11434 的 /v1；api_key 占位即可
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


In [4]:
# ========== 提问：业务场景代码（Stayez 定价），改这里就能问新问题 ==========

# 发给模型的 user 内容保持英文/代码原文（影响回答的字符串不翻译）
question = """
Explain this pricing calculation for my Stayez app:
def calculate_total(base_price, nights, discount_threshold=7, discount_pct=0.10):
    subtotal = base_price * nights
    if nights >= discount_threshold:
        subtotal -= (subtotal * discount_pct)
    return round(subtotal, 2)
"""


In [ ]:
# ========== 路径 A：用 Groq 云端模型流式回答 ==========

# 打印当前调用的云端模型名，方便对照日志
print(f"Calling Cloud Model ({MODEL_CLOUD})...\n")

# 发起流式 Chat Completions：messages 含 system + user
stream = groq_client.chat.completions.create(
    model=MODEL_CLOUD,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ],
    stream=True
)

# 累加完整回答文本
full_response = ""
# 先显示空 Markdown，拿到可更新的 handle（display_id）
display_handle = display(Markdown(""), display_id=True)
# 逐块读取流：把增量拼进 full_response，并原地 update
for chunk in stream:
    txt = chunk.choices[0].delta.content or ""
    full_response += txt
    display_handle.update(Markdown(full_response))


In [ ]:
# ========== 路径 B：用本地 Ollama 模型一次性回答（非流式） ==========

# 打印当前调用的本地模型名
print(f"\nCalling Local Model ({MODEL_LOCAL})...\n")

# 非流式调用：等整段生成完再取 message.content
response = ollama_client.chat.completions.create(
    model=MODEL_LOCAL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]
)

# 把完整回答渲染为 Markdown 展示
display(Markdown(response.choices[0].message.content))
